# Impedance ML Engineering — Session 16
## Use history without leaking the future
**Core: 60 minutes · Temporal modeling · Optional extension: 20–40 minutes**

## Why this matters
How does forecasting later visits for known patients differ from generalizing to entirely new patients?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Specify known-patient versus unseen-patient forecasting estimands.
- Construct lag/change features using strictly prior observations.
- Apply an outcome-availability cutoff in temporal validation.
- Compare forecasts by horizon and discuss sparse sequences.

## R-to-Python / prerequisite bridge
`lag`/`diff` → grouped `.shift`/`.diff` after sorting; a prediction timestamp alone does not prove its outcome was available for training.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Forecasting needs a time-of-availability contract for inputs AND training outcomes. A pair predicted at time t may have its target observed much later; training on that target before it was available leaks the future. A grouped lag must follow time sorting and remain within patient.

Known-patient forecasting can legitimately use earlier observations from the same patient. That answers a different question from predicting in new patients. Sparse follow-up also changes horizon/returner coverage across evaluation windows.

Use `.sort_values(["patient_id","days_since_implant"])`, `.groupby("patient_id")[column].shift(1)`, `.diff()`, and validated merges for target-time audits. Keep future actual timing outside predictors; planned horizon is known, jitter is not.

## Task 1 — Define two forecasting settings (5 min)
Write the Session 02 unseen-patient question and a new known-patient later-time question. Restrict this temporal exercise to the 72 development patients; reserved test patients remain untouched. State which patient overlap is legitimate in known-patient forecasting and what future information is still prohibited.

## Task 2 — Order observed visits (6 min)
Load visits/features for development patients, sort by patient and actual day, and validate unique patient/window keys. Predict maximum sequence length (four) and why a large recurrent network is poorly supported here. Read target visit dates via a validated one-to-one join for audit/availability only, never as predictor columns.

## Task 3 — Create strictly historical features (8 min)
Within each patient calculate previous observed diagonal mean, previous actual day, last change, and elapsed days since the prior visit using shift/diff. At current visit a change from prior to current is allowed; future target values are not. Baseline rows have missing history. Check a single patient manually.

## Task 4 — Audit a temporal leak (6 min)
Inspect the suggestion `features.groupby("patient_id")["diag_mean_kohm"].transform("mean")` as a history predictor. Explain how it uses future visits, even if rows are later sorted. Describe a shifted expanding mean if needed. Audit whether a training pair’s target occurs after the supposed deployment cutoff.

## Task 5 — Build an outcome-available temporal split (8 min)
Choose elapsed-day cutoff 45 as a teaching pseudo-deployment time. Train only development-patient pairs whose target measurement day≤45. Validate pairs with current day≥45 and later targets (normally month 3→6); exclude pairs straddling the cutoff. Report included/excluded counts and group overlap. The cutoff is relative to implantation, not a real shared calendar deployment date.

## Task 6 — Compare simple forecasting recipes (10 min)
Using this temporal split, compare persistence, original summary ridge, and summary+history ridge with imputation/scaling fitted on the temporal training portion only. Keep alpha 1. Warn that training and validation horizons differ; the result answers a shifted-window question, not the earlier patient-holdout question.

## Task 7 — Inspect coverage and history availability (5 min)
Summarize errors by number of available historical visits and planned horizon. Report patients without later outcomes and explain selection among returners. Do not interpolate missing follow-up or use target-day jitter as a known future feature.

## Task 8 — Write the forecasting contract (4 min)
Record prediction time, availability of every feature and training outcome, patient population, temporal cutoff, and horizon distribution. Explain why pooled metrics from different validation questions should not be ranked in one leaderboard.

## Retrieval practice
For one patient, list what data exist at months 0, 1, 3 and which outcomes could have trained a model by day 45.

## Professional reasoning
1. What could go wrong if the split uses current visit date but ignores target availability?
2. Why can patient overlap be valid in one estimand and leakage in another?
3. What extra data density would make a GRU or temporal CNN a meaningful comparison?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Build a small sequence tensor with padded visits, explicit lengths, and only past/current observations, then run a GRU forward pass to inspect shape. With 3–4 visits per patient, treat this as architecture practice rather than evidence for a sequence model.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)